In [0]:
# MAGIC %md
# MAGIC # 01 — Ingestion API ERP → Volume (version simple)
# MAGIC Lit l'API ERP AgriMeca (Azure Container Apps) et dépose chaque page JSON dans
# MAGIC `/Volumes/supplychain/landing/erp/<entité>/ingest_date=AAAAMMJJ/run_id=<id>/page_000001.json`.
# MAGIC Les secrets viennent de Key Vault via le scope `erp` : rien n'est écrit en clair.
 
# COMMAND ----------

In [0]:
# cellule 1 configuration
import json
import time
import uuid
from datetime import datetime, timezone
import requests

BASE_URL = dbutils.secrets.get ("erp", "erp-base-url") # récupération de secret scope
CLIENT_ID = "agrimeca-data"
CLIENT_SECRET = dbutils.secrets.get ("erp", "erp-client-secret") # lire le mot de passe dans Keyvault en passant par le secret scope
LANDING = "/Volumes/supplychain/landing/erp"

print(BASE_URL) # affichage de la base url, databricks masque le secret



In [0]:
# cellule 2 - jeton OAuth (valable 15 minutes)
def get_token() -> str:
    r = requests.post(
        f"{BASE_URL}/auth/token",
        json={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET},
        timeout=30,
    )
    r.raise_for_status()  # lève une exception si HTTP a une erreur (code entre 400 et 599)
    return r.json()["access_token"]


headers = {"Authorization": f"Bearer {get_token()}"}
print("Jeton obtenu")

### VERIFICATION DE LA DISPONIBILITE DE L'API 

In [0]:
print("finit par / :", BASE_URL.endswith("/"))
print("longueur    :", len(BASE_URL))
print(" ".join(BASE_URL))   # affiche l'URL lettre par lettre, ce qui contourne le masquage

In [0]:
r = requests.get(f"{BASE_URL}/docs", timeout=60)
print(r.status_code)
print(r.text[:300])

In [0]:
# cellule 3 - premier appel pour observer la réponse
r= requests.get(f"{BASE_URL}/v1/suppliers",headers=headers, params={"limit":5}, timeout=30)
r.raise_for_status()
page = r.json()

print("meta : ", page["meta"])
print("paging : ", page["paging"])
print("1er  fournisseur :", json.dumps(page["data"][0], indent=2, ensure_ascii=False))

In [0]:
# cellule 4 - pagination et ecriture de chaque page dans le Volume
def get_page(entity:str) -> None:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    folder = f"{LANDING}/{entity}/ingest_date={run_id[:8]}/run_id={run_id}"
    dbutils.fs.mkdirs(folder)

get_page("suppliers")